<a id="top"></a>

# Demo: tool, or no tool?

```yaml
title: "Demo: tool, or no tool?"
keywords: tool-or-no-tool, when to add a tool, model-alone, calculator, current_time, architectural decision, affordances, teacher demo
estimated duration: 12
```

> **Lesson:** L05. Teacher demo — Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L05/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: dry-run before class. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` cannot represent `tool_use` / `tool_result` blocks. L05 registers tools and watches the model *choose* and *call* them, so these demos call the raw SDK directly (exactly as [L04](../L04/L0401_intro.md) introduced). The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. L05 changes the *client*, not where secrets come from.
>
> The point to land: **adding a tool is an architectural decision, not a convenience.** The same task can be answered model-alone or with a tool — the right call depends on the *kind* of question. We run three tasks model-alone first, *then* with a tool, and name why.

## Contents

- [1. Setup — two clean tools and three tasks](#1-setup--two-clean-tools-and-three-tasks)
- [2. Task A — the model has it cold (no tool needed)](#2-task-a--the-model-has-it-cold-no-tool-needed)
- [3. Task B — data the model can't have (tool warranted)](#3-task-b--data-the-model-cant-have-tool-warranted)
- [4. Task C — the subtle one: bad at it, but will try anyway](#4-task-c--the-subtle-one-bad-at-it-but-will-try-anyway)
- [5. Takeaways](#5-takeaways)

## 1. Setup — two clean tools and three tasks

A `calculator` and a `current_time` tool, both with clean designs (we save the *design* walkthrough for [L0505](L0505_lecture.ipynb) — here they are just props). Three tasks span the decision: **A** the model owns cold, **B** the model can't have, **C** borderline arithmetic. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
from datetime import UTC, datetime
from zoneinfo import ZoneInfo

import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # Safe ONLY because the whitelist above blocks names, attributes, and calls.
    return str(eval(expression))


def current_time(tz: str) -> str:
    """Return the current wall-clock time in an IANA timezone, e.g. 'Asia/Tokyo'."""
    return datetime.now(UTC).astimezone(ZoneInfo(tz)).isoformat(timespec="seconds")


CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression and return the exact result. Use whenever "
        "the user asks for the result of a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '18374 * 92431'."}},
        "required": ["expression"],
    },
}
CURRENT_TIME_TOOL: dict[str, object] = {
    "name": "current_time",
    "description": (
        "Return the current time in a given IANA timezone. Use whenever the user asks for "
        "the current or real-time clock anywhere — the model has no clock of its own."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"tz": {"type": "string", "description": "IANA tz name, e.g. 'Asia/Tokyo'."}},
        "required": ["tz"],
    },
}


def show_turn(resp) -> None:
    """Print a model response's blocks: text, and any tool_use (name + args).

    Example output:
        stop_reason: tool_use
          text      'Let me look that up.'
          tool_use  lookup_user(query='alex')  id=toolu_01...
    """
    print("stop_reason:", resp.stop_reason)
    for block in resp.content:
        if block.type == "tool_use":
            args = ", ".join(f"{k}={v!r}" for k, v in block.input.items())
            print(f"  tool_use  {block.name}({args})  id={block.id}")
        else:
            text = getattr(block, "text", "")
            print(f"  text      {text!r}")


def called_tools(resp) -> list[str]:
    """Names of the tools the model proposed in this turn (possibly empty)."""
    return [b.name for b in resp.content if b.type == "tool_use"]


TASK_A = "What is the capital of France?"  # model owns it cold -> NO tool
TASK_B = "What is the current time in Tokyo right now?"  # can't be memorized -> tool
TASK_C = "What is 18,374 multiplied by 92,431?"  # bad at it, but will try -> tool
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. Task A — the model has it cold (no tool needed)

Run Task A model-alone: instant, free, correct. Then register the calculator and run it again — the model usually answers *without* calling the tool. A registered tool is an **option, not an obligation**; here it would be pure overhead and a wrong-tool option to pick by mistake.

In [ ]:
alone = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": TASK_A}],
)
print("=== Task A, model-alone ===")
show_turn(alone)

withtool = client.messages.create(
    model=MODEL,
    max_tokens=200,
    tools=[CALCULATOR_TOOL],
    messages=[{"role": "user", "content": TASK_A}],
)
print("\n=== Task A, calculator registered ===")
show_turn(withtool)
print("tools called:", called_tools(withtool), "(expect none — a tool here is overhead)")

[↑ Back to top](#top)

## 3. Task B — data the model can't have (tool warranted)

Run Task B model-alone: the model refuses, hedges (*'I don't have access to real-time data'*), or hallucinates a time. Then register `current_time`: it calls the tool and gets a real answer. This is the **'data the model can't have memorized'** signal.

In [ ]:
alone = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": TASK_B}],
)
print("=== Task B, model-alone (hedge / guess expected) ===")
show_turn(alone)

withtool = client.messages.create(
    model=MODEL,
    max_tokens=200,
    tools=[CURRENT_TIME_TOOL],
    messages=[{"role": "user", "content": TASK_B}],
)
print("\n=== Task B, current_time registered ===")
show_turn(withtool)
print("tools called:", called_tools(withtool), "(expect ['current_time'])")

[↑ Back to top](#top)

## 4. Task C — the subtle one: bad at it, but will try anyway

Run Task C model-alone **three times**. The answers are often inconsistent across runs, and sometimes wrong — the model *will* attempt the arithmetic, confidently. Then register the calculator: a deterministic, correct answer. Task C is the subtle signal: the tool fills a gap the **designer** knows about, not one the model knows about.

In [ ]:
print("=== Task C, model-alone x3 (watch for drift) ===")
for i in range(3):
    r = client.messages.create(
        model=MODEL,
        max_tokens=120,
        messages=[{"role": "user", "content": TASK_C}],
    )
    text = "".join(b.text for b in r.content if b.type == "text")
    print(f"  run {i + 1}: {text.strip()[:80]}")

print("\nground truth (Python):", 18374 * 92431)

withtool = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[CALCULATOR_TOOL],
    messages=[{"role": "user", "content": TASK_C}],
)
print("\n=== Task C, calculator registered ===")
show_turn(withtool)

[↑ Back to top](#top)

## 5. Takeaways

- The four **'tool warranted'** signals (from [objectives.md](../../../../docs/origin/lesson_roadmaps/L05/objectives.md)): data the model can't memorize (B), precise computation it's bad at (C), side effects (foreshadow [L0509](L0509_lecture.ipynb)), verification against ground truth. Name each as it appears.
- The **'no tool needed'** case (A): the model already owns it reliably, and the round-trip isn't worth a marginal gain. A tool here wastes tokens and adds a wrong-tool failure mode.
- **Adding a tool is designing the agent's affordances** — an architectural decision. *More tools ≠ more capable agent* (reinforce [L01](../L01/L0101_intro.md) on context-window cost; don't re-teach).
- The deliverable is **one sentence defending each call** — naming *why* is the skill the [L0504 lab](L0504_lab_empty.ipynb) drills.
- Next: [L0505](L0505_lecture.ipynb) — once you've decided *yes, a tool*, the description decides whether the model uses it well.

[↑ Back to top](#top)